Everything runs on the CPU because my GPU is too weak to hold all the code, and it would cause an out-of-memory error and a complete system crash.

Configuration

In [ ]:
import os, sys, yaml, torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.getcwd())

with open('configs/hyperparams.yaml', 'r') as f:
    config = yaml.safe_load(f)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = os.path.join(os.getcwd(), 'data')
RESULTS_DIR = 'results'
CHECKPOINTS_DIR = 'checkpoints'
NUM_RUNS = 3

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'tables'), exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(os.path.join(CHECKPOINTS_DIR, 'pretrained'), exist_ok=True)
os.makedirs(os.path.join(CHECKPOINTS_DIR, 'source_trajectories'), exist_ok=True)

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

Validation de l'hypothèse (P) - Figure 3

In [ ]:
import torch
import glob as glob_module
from models.mlp import TwoLayerMLP
from training.trainer import train_model
from evaluation.cosine_similarity import evaluate_assumption_P, plot_cosine_similarities
from experiments.exp1_random_init import get_mnist_loaders

print('=== Hypothesis (P) Evaluation - 2-MLP on MNIST ===')
train_loader_mnist, val_loader_mnist = get_mnist_loaders(
    os.path.join(DATA_ROOT, 'mnist')
)

In [ ]:
fig3_results = {}

for hidden_dim in [8, 16, 32, 64, 128]:
    print(f'  Hidden dim = {hidden_dim}')
    model_kwargs = {'input_dim': 784, 'hidden_dim': hidden_dim, 'output_dim': 10}

    ckpt_dir_0 = os.path.join(CHECKPOINTS_DIR, f'mlp_h{hidden_dim}_run0')
    ckpt_dir_1 = os.path.join(CHECKPOINTS_DIR, f'mlp_h{hidden_dim}_run1')

    torch.manual_seed(0)
    m1 = TwoLayerMLP(**model_kwargs)
    m1_trained, _ = train_model(
        m1, train_loader_mnist, None,
        epochs=50, lr=0.01, momentum=0.9, weight_decay=1e-4,
        device=DEVICE, verbose=False,
        save_checkpoints=True, checkpoint_dir=ckpt_dir_0
    )

    torch.manual_seed(42)
    m2 = TwoLayerMLP(**model_kwargs)
    m2_trained, _ = train_model(
        m2, train_loader_mnist, None,
        epochs=50, lr=0.01, momentum=0.9, weight_decay=1e-4,
        device=DEVICE, verbose=False,
        save_checkpoints=True, checkpoint_dir=ckpt_dir_1
    )

    def load_traj(ckpt_dir, step=10):
        ckpts = sorted(glob_module.glob(os.path.join(ckpt_dir, 'epoch_*.pth')))
        return [torch.load(p, map_location='cpu') for p in ckpts[::step]]

    traj1 = load_traj(ckpt_dir_0, step=10)
    traj2 = load_traj(ckpt_dir_1, step=10)

    min_len = min(len(traj1), len(traj2))
    if min_len > 1:
        traj1 = traj1[:min_len]
        traj2 = traj2[:min_len]
        cos_perm, cos_id = evaluate_assumption_P(traj1, traj2, architecture='mlp')
        fig3_results[f'{hidden_dim} (permuted)'] = (cos_perm, cos_id)
        print(f'    Average cos_sim (permuted): {sum(cos_perm)/len(cos_perm):.4f}')
    else:
        print(f'    Not enough checkpoints for h={hidden_dim}')

if fig3_results:
    fig3 = plot_cosine_similarities(
        results=fig3_results,
        title='Cosine Similarity - 2-MLP MNIST (Fig. 3a)',
        save_path=os.path.join(RESULTS_DIR, 'figures', 'fig3_cosine_similarity.png'),
        show=True
    )
    plt.close()
    print('Figure 3 saved.')
else:
    print('No results for Figure 3.')

Pré-entrainement CIFAR-10 / Conv8

In [ ]:
import torch
from models.conv8 import Conv8
from training.trainer import train_model
from experiments.exp1_random_init import get_cifar10_loaders

cifar10_cfg = config['cifar10_conv8']
train_loader_c10, val_loader_c10 = get_cifar10_loaders(
    os.path.join(DATA_ROOT, 'cifar10'),
    batch_size=cifar10_cfg['batch_size']
)

In [ ]:
for run in range(NUM_RUNS):
    ckpt_path = os.path.join(CHECKPOINTS_DIR, 'pretrained', f'cifar10_conv8_run{run}.pth')
    if os.path.exists(ckpt_path):
        print(f'CIFAR-10 checkpoint run{run} already exists, skipping.')
        continue

    print(f'\nCIFAR-10 Conv8 pretraining run {run+1}/{NUM_RUNS}...')
    torch.manual_seed(run * 100)
    model = Conv8(num_classes=10)

    trained_model, history = train_model(
        model=model,
        train_loader=train_loader_c10,
        val_loader=val_loader_c10,
        epochs=cifar10_cfg['epochs'],
        lr=cifar10_cfg['lr'],
        momentum=cifar10_cfg['momentum'],
        weight_decay=cifar10_cfg['weight_decay'],
        lr_schedule=[30, 45],
        device=DEVICE,
        verbose=True
    )
    torch.save(trained_model.state_dict(), ckpt_path)
    print(f'Saved: {ckpt_path}')
    print(f'Final val acc: {history["val_acc"][-1]:.2f}%')

Pré-entrainement ImageNet / ResNet-18

In [ ]:
import torch
import gc
import torchvision.models as tv_models
from models.resnet import get_resnet18

IMAGENET_AVAILABLE = os.path.isdir(os.path.join(DATA_ROOT, 'imagenet', 'train'))

In [ ]:
if IMAGENET_AVAILABLE:
    from training.pretrain import pretrain_on_imagenet
    for run in range(NUM_RUNS):

        torch.cuda.empty_cache()
        gc.collect()

        ckpt_path = os.path.join(CHECKPOINTS_DIR, 'pretrained', f'imagenet_resnet18_run{run}.pth')
        if os.path.exists(ckpt_path):
            print(f'ImageNet checkpoint run{run} already exists, skipping.')
            continue
        print(f'\nImageNet ResNet-18 pretraining run {run+1}/{NUM_RUNS}...')
        pretrain_on_imagenet(
            imagenet_root=os.path.join(DATA_ROOT, 'imagenet'),
            save_path=ckpt_path,
            epochs=90,
            batch_size=256,
            device=DEVICE
        )

        torch.cuda.empty_cache()
        gc.collect()

else:
    print('ImageNet not available — using official torchvision weights.')
    for run in range(NUM_RUNS):

        ckpt_path = os.path.join(CHECKPOINTS_DIR, 'pretrained', f'imagenet_resnet18_run{run}.pth')
        if os.path.exists(ckpt_path):
            print(f'Checkpoint run{run} already exists, skipping.')
            continue
        model = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        if run > 0:
            with torch.no_grad():
                for param in model.parameters():
                    param.add_(torch.randn_like(param) * 0.001)
        torch.save(model.state_dict(), ckpt_path)
        print(f'Torchvision checkpoint saved: {ckpt_path}')

        torch.cuda.empty_cache()
        gc.collect()

Pré-entrainement ImageNet 10% (Figure 8)

In [ ]:
import shutil

ckpt_10pct = os.path.join(CHECKPOINTS_DIR, 'pretrained', 'imagenet_10pct_resnet18.pth')
ckpt_full  = os.path.join(CHECKPOINTS_DIR, 'pretrained', 'imagenet_full_resnet18.pth')

In [ ]:
if IMAGENET_AVAILABLE:
    from training.pretrain import pretrain_on_imagenet
    if not os.path.exists(ckpt_10pct):
        print('ImageNet-10% pretraining...')
        torch.cuda.empty_cache()
        gc.collect()

        pretrain_on_imagenet(
            imagenet_root=os.path.join(DATA_ROOT, 'imagenet'),
            save_path=ckpt_10pct,
            epochs=90,
            subset_fraction=0.1,
            device=DEVICE
        )
        torch.cuda.empty_cache()
        gc.collect()
    else:
        print('ImageNet-10% checkpoint already exists, skipping.')
else:
    print('ImageNet not available, skipping ImageNet-10%.')

# Copy run0 as full reference if missing
src_run0 = os.path.join(CHECKPOINTS_DIR, 'pretrained', 'imagenet_resnet18_run0.pth')
if not os.path.exists(ckpt_full) and os.path.exists(src_run0):
    shutil.copy(src_run0, ckpt_full)
    print(f'Full checkpoint copied from run0: {ckpt_full}')
elif os.path.exists(ckpt_full):
    print('ImageNet-Full checkpoint already exists, skipping.')
else:
    print('Run0 checkpoint not found to copy full.')

Expérience 1 : Random Init Transfer (MNITS - CIFAR10 - ImageNet)

In [ ]:
from experiments.exp1_random_init import run_mnist_experiment

print('=== Experiment 1a: MNIST ===')
mnist_results = run_mnist_experiment(
    config=config,
    data_root=DATA_ROOT,
    results_dir=RESULTS_DIR,
    checkpoints_dir=CHECKPOINTS_DIR,
    device='cpu', # DEVICE
    num_runs=NUM_RUNS
)
np.save(os.path.join(RESULTS_DIR, 'exp1_mnist_results.npy'), mnist_results, allow_pickle=True)
print('Experiment 1a completed.')

In [ ]:
from experiments.exp1_random_init import run_cifar10_experiment

print('=== Experiment 1b: CIFAR-10 ===')
cifar10_results = run_cifar10_experiment(
    config=config,
    data_root=DATA_ROOT,
    results_dir=RESULTS_DIR,
    checkpoints_dir=CHECKPOINTS_DIR,
    device='cpu', # DEVICE
    num_runs=NUM_RUNS
)
np.save(os.path.join(RESULTS_DIR, 'exp1_cifar10_results.npy'), cifar10_results, allow_pickle=True)
print('Experiment 1b completed.')

In [ ]:
from experiments.exp1_random_init import run_imagenet_experiment
import gc

print('=== Experiment 1c: ImageNet / ResNet-18 ===')
gc.collect()
imagenet_results = run_imagenet_experiment(
    config=config,
    data_root=DATA_ROOT,
    results_dir=RESULTS_DIR,
    checkpoints_dir=CHECKPOINTS_DIR,
    device='cpu', # DEVICE
    num_runs=NUM_RUNS
)
gc.collect()
np.save(os.path.join(RESULTS_DIR, 'exp1_imagenet_results.npy'), imagenet_results, allow_pickle=True)
print('Experiment 1c completed.')

Expérience 2 : Cars - CUB - CIFAR10/CIFAR100

In [ ]:
# 2a - Cars
from experiments.exp2_pretrained_init import run_cars_experiment

print('=== Experiment 2a: ImageNet -> Stanford Cars ===')
gc.collect()
cars_results = run_cars_experiment(
    config=config,
    data_root=DATA_ROOT,
    checkpoints_dir=CHECKPOINTS_DIR,
    results_dir=RESULTS_DIR,
    device='cpu', # DEVICE
    num_runs=NUM_RUNS
)
gc.collect()
np.save(os.path.join(RESULTS_DIR, 'exp2_cars_results.npy'), cars_results, allow_pickle=True)
print('Experiment 2a completed.')

In [ ]:
# 2b - CUB
from experiments.exp2_pretrained_init import run_cub_experiment

print('=== Experiment 2b: ImageNet -> CUB-200-2011 ===')
gc.collect()
cub_results = run_cub_experiment(
    config=config,
    data_root=DATA_ROOT,
    checkpoints_dir=CHECKPOINTS_DIR,
    results_dir=RESULTS_DIR,
    device='cpu', # DEVICE
    num_runs=NUM_RUNS
)
gc.collect()
np.save(os.path.join(RESULTS_DIR, 'exp2_cub_results.npy'), cub_results, allow_pickle=True)
print('Experiment 2b completed.')

In [ ]:
# 2c - CIFAR10 / CIFAR100
from experiments.exp2_pretrained_init import run_cifar10_to_cifar100_experiment

print('=== Experiment 2c: CIFAR-10 -> CIFAR-100 ===')
gc.collect()
c10_c100_results = run_cifar10_to_cifar100_experiment(
    config=config,
    data_root=DATA_ROOT,
    checkpoints_dir=CHECKPOINTS_DIR,
    results_dir=RESULTS_DIR,
    device='cpu', # DEVICE
    num_runs=NUM_RUNS
)
gc.collect()
np.save(os.path.join(RESULTS_DIR, 'exp2_cifar10_to_cifar100_results.npy'), c10_c100_results, allow_pickle=True)
print('Experiment 2c completed.')

Expérience 3 : Accelerated Training

In [ ]:
import torch
from models.conv8 import Conv8
from transfer.fgmt import FGMT
from transfer.baselines import NaiveTransfer
from transfer.trajectory import LinearTrajectory
from training.trainer import train_model, evaluate_state_dict
from experiments.exp1_random_init import get_cifar10_loaders
from experiments.exp3_accelerated_training import (
    find_best_timestep,
    run_subsequent_training,
    plot_subsequent_training
)

print('=== Experiment 3: Accelerated Training (CIFAR-10) ===')

cifar10_cfg = config['cifar10_conv8']
train_loader_c10, val_loader_c10 = get_cifar10_loaders(
    os.path.join(DATA_ROOT, 'cifar10'),
    batch_size=cifar10_cfg['batch_size']
)
model_kwargs_c10 = {'num_classes': 10}

torch.manual_seed(0)
model_src = Conv8(num_classes=10)
theta1_0 = {k: v.clone() for k, v in model_src.state_dict().items()}
model_src_trained, _ = train_model(
    model_src, train_loader_c10, val_loader_c10,
    epochs=cifar10_cfg['epochs'],
    lr=cifar10_cfg['lr'],
    momentum=cifar10_cfg['momentum'],
    weight_decay=cifar10_cfg['weight_decay'],
    lr_schedule=[30, 45],
    device='cpu', verbose=False # DEVICE
)
theta1_T = {k: v.clone() for k, v in model_src_trained.state_dict().items()}
source_traj = LinearTrajectory(theta1_0, theta1_T, length=cifar10_cfg['trajectory_length'])

torch.manual_seed(42)
theta2_0 = {k: v.clone() for k, v in Conv8(num_classes=10).state_dict().items()}

fgmt = FGMT(
    model_class=Conv8,
    model_kwargs=model_kwargs_c10,
    architecture='conv',
    device= 'cpu' # DEVICE
)
fgmt_params, _ = fgmt.transfer(source_traj, theta2_0, train_loader_c10, verbose=True)

naive = NaiveTransfer(architecture='conv')
naive_params, _ = naive.transfer(source_traj, theta2_0)

_, _, best_sd_fgmt = find_best_timestep(
    fgmt_params, Conv8, model_kwargs_c10, val_loader_c10, device='cpu' # DEVICE
)
_, _, best_sd_naive = find_best_timestep(
    naive_params, Conv8, model_kwargs_c10, val_loader_c10, device='cpu' # DEVICE
)

subsequent_cfg = {**cifar10_cfg, 'epochs': 60}

std_accs = run_subsequent_training(
    {k: v.clone() for k, v in Conv8(num_classes=10).state_dict().items()},
    Conv8, model_kwargs_c10,
    train_loader_c10, val_loader_c10,
    subsequent_cfg, device='cpu', label='Standard Training' # DEVICE
)
fgmt_accs = run_subsequent_training(
    best_sd_fgmt, Conv8, model_kwargs_c10,
    train_loader_c10, val_loader_c10,
    subsequent_cfg, device='cpu', label='FGMT Transfer' # DEVICE
)
naive_accs = run_subsequent_training(
    best_sd_naive, Conv8, model_kwargs_c10,
    train_loader_c10, val_loader_c10,
    subsequent_cfg, device='cpu', label='Naive Transfer' # DEVICE
)

plot_subsequent_training(
    val_accs_per_method={
        'Standard Training': std_accs,
        'FGMT Transfer': fgmt_accs,
        'Naive Transfer': naive_accs
    },
    title='CIFAR-10 (Conv8) - Subsequent Training (Fig. 6a)',
    save_path=os.path.join(RESULTS_DIR, 'figures', 'fig6a_cifar10.png'),
    show=True
)
plt.close()
print('Experiment 3 completed.')

Expérience 4a : Loss Paysage

In [ ]:
import torch
from models.conv8 import Conv8
from transfer.fgmt import FGMT
from transfer.trajectory import LinearTrajectory
from permutation.alignment import align_parameters
from permutation.symmetry import apply_permutation_to_state_dict
from training.finetune import get_cifar100_subset_loaders, finetune_model
from training.trainer import evaluate_model
from evaluation.loss_landscape import (
    compute_accuracy_landscape_2d,
    compute_directions_from_models,
    plot_loss_landscape
)
from experiments.exp3_accelerated_training import find_best_timestep

print('=== Experiment 4a: Loss Landscape (Fig. 7) ===')

cifar10_to_100_cfg = config['cifar10_to_cifar100']
train_loader_c100, val_loader_c100, _ = get_cifar100_subset_loaders(
    os.path.join(DATA_ROOT, 'cifar100'),
    num_classes=10,
    batch_size=cifar10_to_100_cfg['batch_size']
)

ckpt_src = os.path.join(CHECKPOINTS_DIR, 'pretrained', 'cifar10_conv8_run0.pth')
ckpt_tgt = os.path.join(CHECKPOINTS_DIR, 'pretrained', 'cifar10_conv8_run1.pth')

if os.path.exists(ckpt_src) and os.path.exists(ckpt_tgt):
    model_kwargs_10 = {'num_classes': 10}

    model_src = Conv8(num_classes=10)
    model_src.load_state_dict(torch.load(ckpt_src, map_location='cpu'), strict=False)
    theta1_0 = {k: v.clone() for k, v in model_src.state_dict().items()}
    model_src_ft, _ = finetune_model(
        model_src, train_loader_c100, val_loader_c100, 10,
        epochs=cifar10_to_100_cfg['epochs'],
        lr=cifar10_to_100_cfg['lr'],
        momentum=cifar10_to_100_cfg['momentum'],
        weight_decay=cifar10_to_100_cfg['weight_decay'],
        device='cpu', verbose=False # DEVICE
    )
    theta1_T = {k: v.clone() for k, v in model_src_ft.state_dict().items()}

    model_tgt = Conv8(num_classes=10)
    model_tgt.load_state_dict(torch.load(ckpt_tgt, map_location='cpu'), strict=False)
    theta2_0 = {k: v.clone() for k, v in model_tgt.state_dict().items()}
    model_tgt_ft, _ = finetune_model(
        model_tgt, train_loader_c100, val_loader_c100, 10,
        epochs=cifar10_to_100_cfg['epochs'],
        lr=cifar10_to_100_cfg['lr'],
        momentum=cifar10_to_100_cfg['momentum'],
        weight_decay=cifar10_to_100_cfg['weight_decay'],
        device='cpu', verbose=False # DEVICE
    )
    theta2_T = {k: v.clone() for k, v in model_tgt_ft.state_dict().items()}

    perm_gitrebasin = align_parameters(theta1_T, theta2_T, architecture='conv')
    theta_source_permuted = apply_permutation_to_state_dict(
        theta1_T, perm_gitrebasin, 'conv'
    )

    source_traj = LinearTrajectory(
        theta1_0, theta1_T,
        length=cifar10_to_100_cfg['trajectory_length']
    )
    fgmt = FGMT(
        model_class=Conv8,
        model_kwargs=model_kwargs_10,
        architecture='conv',
        device='cpu' # DEVICE
    )
    fgmt_params, _ = fgmt.transfer(source_traj, theta2_0, train_loader_c100, verbose=False)

    _, _, best_sd_fgmt = find_best_timestep(
        fgmt_params, Conv8, model_kwargs_10, val_loader_c100, device='cpu' # DEVICE
    )

    model_fgmt = Conv8(num_classes=10)
    model_fgmt.load_state_dict(best_sd_fgmt, strict=False)
    model_fgmt_ft, _ = finetune_model(
        model_fgmt, train_loader_c100, val_loader_c100, 10,
        epochs=cifar10_to_100_cfg['epochs'],
        lr=cifar10_to_100_cfg['lr'],
        momentum=cifar10_to_100_cfg['momentum'],
        weight_decay=cifar10_to_100_cfg['weight_decay'],
        device='cpu', verbose=False # DEVICE
    )
    theta_fgmt_ft = {k: v.clone() for k, v in model_fgmt_ft.state_dict().items()}

    dir_u, dir_v = compute_directions_from_models(
        theta2_T, theta_source_permuted, theta_fgmt_ft
    )
    alphas, betas, acc_grid = compute_accuracy_landscape_2d(
        center=theta2_T,
        direction_u=dir_u,
        direction_v=dir_v,
        model_class=Conv8,
        model_kwargs=model_kwargs_10,
        dataloader=val_loader_c100,
        grid_size=15,
        grid_range=1.0,
        device='cpu' # DEVICE
    )
    fig7 = plot_loss_landscape(
        alphas, betas, acc_grid,
        title='Loss Landscape - CIFAR-10→CIFAR-100 (Fig. 7)',
        save_path=os.path.join(RESULTS_DIR, 'figures', 'fig7_cifar10_to_cifar100.png'),
        cmap='RdYlGn',
        show=True
    )
    plt.close()
    print('Figure 7 saved.')
else:
    print('CIFAR-10 checkpoints missing. Run previous cell first.')

Experience 4b : Héritage de Généralisation

In [ ]:
# Method 1: Standard (ImageNet-Full)
from experiments.exp4_inheritance import run_method1_standard_full

ckpt_full = os.path.join(CHECKPOINTS_DIR, 'pretrained', 'imagenet_full_resnet18.pth')

if os.path.exists(ckpt_full):
    print('=== Experiment 4b - Method 1: Standard (ImageNet-Full) ===')
    method1_results = run_method1_standard_full(
        config=config,
        data_root=DATA_ROOT,
        checkpoints_dir=CHECKPOINTS_DIR,
        results_dir=RESULTS_DIR,
        device='cpu'
    )
    print('Method 1 completed.')
else:
    print('ImageNet-Full checkpoint missing. Skipping.')

In [ ]:
# Method 2: Standard (ImageNet-10%)
from experiments.exp4_inheritance import run_method2_standard_10pct

ckpt_10pct = os.path.join(CHECKPOINTS_DIR, 'pretrained', 'imagenet_10pct_resnet18.pth')

if os.path.exists(ckpt_10pct):
    print('=== Experiment 4b - Method 2: Standard (ImageNet-10%) ===')
    method2_results = run_method2_standard_10pct(
        config=config,
        data_root=DATA_ROOT,
        checkpoints_dir=CHECKPOINTS_DIR,
        results_dir=RESULTS_DIR,
        device='cpu'
    )
    print('Method 2 completed.')
else:
    print("ImageNet-10% checkpoint missing. Skipping.")

In [ ]:
# Method 3: FGMT (10% -> Full)
from experiments.exp4_inheritance import run_method3_fgmt_10pct_to_full

ckpt_10pct = os.path.join(CHECKPOINTS_DIR, 'pretrained', 'imagenet_10pct_resnet18.pth')
ckpt_full = os.path.join(CHECKPOINTS_DIR, 'pretrained', 'imagenet_full_resnet18.pth')

if os.path.exists(ckpt_10pct) and os.path.exists(ckpt_full):
    print('=== Experiment 4b - Method 3: FGMT (10% -> Full) ===')
    method3_results = run_method3_fgmt_10pct_to_full(
        config=config,
        data_root=DATA_ROOT,
        checkpoints_dir=CHECKPOINTS_DIR,
        results_dir=RESULTS_DIR,
        device='cpu'
    )
    print('Method 3 completed.')
else:
    print('Checkpoints missing. Skipping.')

In [ ]:
# Method 4: FGMT (10% -> 10%)
from experiments.exp4_inheritance import run_method4_fgmt_10pct_to_10pct

ckpt_10pct = os.path.join(CHECKPOINTS_DIR, 'pretrained', 'imagenet_10pct_resnet18.pth')

if os.path.exists(ckpt_10pct):
    print('=== Experiment 4b - Method 4: FGMT (10% -> 10%) ===')
    method4_results = run_method4_fgmt_10pct_to_10pct(
        config=config,
        data_root=DATA_ROOT,
        checkpoints_dir=CHECKPOINTS_DIR,
        results_dir=RESULTS_DIR,
        device='cpu'
    )
    print('Method 4 completed.')
else:
    print("ImageNet-10% checkpoint missing. Skipping.")

Figure 4 : Trajectoire Linéaire vs Réelle

In [ ]:
# Loading and source training
import torch
import glob as glob_module
from models.conv8 import Conv8
from training.trainer import train_model
from experiments.exp1_random_init import get_cifar10_loaders

print('=== Figure 4: Linear vs. Actual Trajectory ===')  

cifar10_cfg = config['cifar10_conv8']
T = cifar10_cfg['trajectory_length']
train_loader_c10, val_loader_c10 = get_cifar10_loaders(
    os.path.join(DATA_ROOT, 'cifar10'),
    batch_size=cifar10_cfg['batch_size']
)
model_kwargs_c10 = {'num_classes': 10}

torch.manual_seed(7)
model_src = Conv8(num_classes=10)
theta1_0 = {k: v.clone() for k, v in model_src.state_dict().items()}

ckpt_dir_fig4 = os.path.join(CHECKPOINTS_DIR, 'fig4_source')
_, _ = train_model(
    model_src, train_loader_c10, val_loader_c10,
    epochs=T,
    lr=cifar10_cfg['lr'],
    momentum=cifar10_cfg['momentum'],
    weight_decay=cifar10_cfg['weight_decay'],
    lr_schedule=[20, 25],
    device='cpu', verbose=False,
    save_checkpoints=True,
    checkpoint_dir=ckpt_dir_fig4
)
theta1_T = {k: v.clone() for k, v in model_src.state_dict().items()}

torch.manual_seed(99)
theta2_0 = {k: v.clone() for k, v in Conv8(num_classes=10).state_dict().items()}

print('Source training completed.')

In [ ]:
# Linear trajectory
from transfer.trajectory import LinearTrajectory
from transfer.fgmt import FGMT
from training.trainer import evaluate_state_dict
import gc

linear_traj = LinearTrajectory(theta1_0, theta1_T, length=T)
fgmt_lin = FGMT(
    model_class=Conv8,
    model_kwargs=model_kwargs_c10,
    architecture='conv',
    device='cpu'
)
lin_params, _ = fgmt_lin.transfer(linear_traj, theta2_0, train_loader_c10, verbose=False)
gc.collect()

lin_accs = []
for sd in lin_params:
    _, acc = evaluate_state_dict(sd, Conv8, model_kwargs_c10, val_loader_c10, device='cpu')
    lin_accs.append(acc)

gc.collect()
print(f'Linear trajectory completed. {len(lin_accs)} timesteps evaluated.')  

In [ ]:
# Actual trajectory
from permutation.alignment import align_parameters
from permutation.symmetry import apply_permutation_to_state_dict
from training.trainer import evaluate_state_dict
import gc

ckpt_files = sorted(glob_module.glob(os.path.join(ckpt_dir_fig4, 'epoch_*.pth')))
actual_traj_sds = [torch.load(p, map_location='cpu') for p in ckpt_files]

actual_accs = []
if len(actual_traj_sds) >= 2:
    actual_traj_full = [theta1_0] + actual_traj_sds
    perm_actual = align_parameters(actual_traj_full[-1], theta2_0, architecture='conv')
    current = {k: v.clone() for k, v in theta2_0.items()}
    for t in range(1, len(actual_traj_full)):
        diff_t = {
            k: actual_traj_full[t][k].float() - actual_traj_full[t-1][k].float()
            for k in theta1_0 if k in actual_traj_full[t]
        }
        perm_diff = apply_permutation_to_state_dict(diff_t, perm_actual, 'conv')
        new_sd = {
            k: current[k] + perm_diff.get(k, torch.zeros_like(current[k]))
            for k in current
        }
        current = new_sd
        _, acc = evaluate_state_dict(
            new_sd, Conv8, model_kwargs_c10, val_loader_c10, device='cpu'
        )
        actual_accs.append(acc)
        gc.collect()

print(f'Actual trajectory completed. {len(actual_accs)} timesteps evaluated.') 

In [ ]:
# Display
fig4, ax4 = plt.subplots(figsize=(8, 5))
ax4.plot(range(1, len(lin_accs)+1), lin_accs,
         color='blue', label='Transferring linear trajectory', linewidth=2)
ax4.fill_between(
    range(1, len(lin_accs)+1),
    [a - 1 for a in lin_accs], [a + 1 for a in lin_accs],
    alpha=0.2, color='blue'
)
if actual_accs:
    ax4.plot(range(1, len(actual_accs)+1), actual_accs,
             color='orange', label='Transferring actual trajectory',
             linewidth=2, linestyle='--')
    ax4.fill_between(
        range(1, len(actual_accs)+1),
        [a - 2 for a in actual_accs], [a + 2 for a in actual_accs],
        alpha=0.2, color='orange'
    )
ax4.set_xlabel('Trajectory Timestep', fontsize=12)
ax4.set_ylabel('Val Accuracy (%)', fontsize=12)
ax4.set_title('Linear vs Actual Trajectory (Conv8 on CIFAR-10) — Fig. 4', fontsize=13)
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, 'figures', 'fig4_linear_vs_actual.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
plt.close()
print('Figure 4 saved.') 

5 - Sauvegardes

In [ ]:
# Cars
from training.finetune import get_stanford_cars_loaders
from models.resnet import get_resnet18
import gc

cars_cfg = config['imagenet_to_cars']
train_loader_cars, val_loader_cars = get_stanford_cars_loaders(
    os.path.join(DATA_ROOT, 'cars'),
    batch_size=cars_cfg['batch_size']
)
model_kwargs_cars = {'num_classes': 196}
subsequent_cfg_cars = {**cars_cfg, 'epochs': 30}

params_dir_cars = os.path.join(RESULTS_DIR, 'cars')
val_accs_cars = {}

std_model = get_resnet18(num_classes=196)
val_accs_cars['Standard Training'] = run_subsequent_training(
    std_model.state_dict(), get_resnet18, model_kwargs_cars,
    train_loader_cars, val_loader_cars, subsequent_cfg_cars, device='cpu', label='Standard Training'
)
gc.collect()

for method_key, method_label in [('fgmt', 'FGMT Transfer'), ('gmt', 'GMT Transfer'),
                                   ('oracle', 'Oracle Transfer'), ('naive', 'Naive Transfer')]:
    params_path = os.path.join(params_dir_cars, f'run0_{method_key}_params.npy')
    if os.path.exists(params_path):
        params = np.load(params_path, allow_pickle=True).tolist()
        _, _, best_sd = find_best_timestep(params, get_resnet18, model_kwargs_cars, val_loader_cars, device='cpu')
        val_accs_cars[method_label] = run_subsequent_training(
            best_sd, get_resnet18, model_kwargs_cars,
            train_loader_cars, val_loader_cars, subsequent_cfg_cars, device='cpu', label=method_label
        )
        gc.collect()
    else:
        print(f'{method_label} params not found. Rerun exp2a with params saved.') 

plot_subsequent_training(
    val_accs_per_method=val_accs_cars,
    title='ImageNet→Cars - Subsequent Training (Fig. 6c)',
    save_path=os.path.join(RESULTS_DIR, 'figures', 'fig6c_imagenet_to_cars.png'),
    show=True
)
plt.close()
print('Figure 6c saved.')  

In [ ]:
# CUB
from training.finetune import get_cub200_loaders
import gc

cub_cfg = config['imagenet_to_cub']
train_loader_cub, val_loader_cub = get_cub200_loaders(
    os.path.join(DATA_ROOT, 'cub200'),
    batch_size=cub_cfg['batch_size']
)
model_kwargs_cub = {'num_classes': 200}
subsequent_cfg_cub = {**cub_cfg, 'epochs': 30}

params_dir_cub = os.path.join(RESULTS_DIR, 'cub200')
val_accs_cub = {}

std_model = get_resnet18(num_classes=200)
val_accs_cub['Standard Training'] = run_subsequent_training(
    std_model.state_dict(), get_resnet18, model_kwargs_cub,
    train_loader_cub, val_loader_cub, subsequent_cfg_cub, device='cpu', label='Standard Training'
)
gc.collect()

for method_key, method_label in [('fgmt', 'FGMT Transfer'), ('gmt', 'GMT Transfer'),
                                   ('oracle', 'Oracle Transfer'), ('naive', 'Naive Transfer')]:
    params_path = os.path.join(params_dir_cub, f'run0_{method_key}_params.npy')
    if os.path.exists(params_path):
        params = np.load(params_path, allow_pickle=True).tolist()
        _, _, best_sd = find_best_timestep(params, get_resnet18, model_kwargs_cub, val_loader_cub, device='cpu')
        val_accs_cub[method_label] = run_subsequent_training(
            best_sd, get_resnet18, model_kwargs_cub,
            train_loader_cub, val_loader_cub, subsequent_cfg_cub, device='cpu', label=method_label
        )
        gc.collect()
    else:
        print(f'{method_label} params not found. Rerun exp2b with params saved.') 

plot_subsequent_training(
    val_accs_per_method=val_accs_cub,
    title='ImageNet→CUB - Subsequent Training (Fig. 6d)',
    save_path=os.path.join(RESULTS_DIR, 'figures', 'fig6d_imagenet_to_cub.png'),
    show=True
)
plt.close()
print('Figure 6d saved.') 

In [ ]:
# CIFAR10 -> CIFAR 100
from experiments.exp3_accelerated_training import find_best_timestep, run_subsequent_training, plot_subsequent_training
from training.finetune import get_cifar100_subset_loaders
from models.conv8 import Conv8
import gc

cifar10_to_100_cfg = config['cifar10_to_cifar100']
train_loader_c100, val_loader_c100, _ = get_cifar100_subset_loaders(
    os.path.join(DATA_ROOT, 'cifar100'),
    num_classes=10,
    batch_size=cifar10_to_100_cfg['batch_size']
)
model_kwargs_c100 = {'num_classes': 10}
subsequent_cfg = {**cifar10_to_100_cfg, 'epochs': 60}

params_dir = os.path.join(RESULTS_DIR, 'cifar10_to_cifar100')
val_accs_c100 = {}

std_model = Conv8(num_classes=10)
val_accs_c100['Standard Training'] = run_subsequent_training(
    std_model.state_dict(), Conv8, model_kwargs_c100,
    train_loader_c100, val_loader_c100, subsequent_cfg, device='cpu', label='Standard Training'
)
gc.collect()

for method_key, method_label in [('fgmt', 'FGMT Transfer'), ('gmt', 'GMT Transfer'),
                                   ('oracle', 'Oracle Transfer'), ('naive', 'Naive Transfer')]:
    params_path = os.path.join(params_dir, f'run0_{method_key}_params.npy')
    if os.path.exists(params_path):
        params = np.load(params_path, allow_pickle=True).tolist()
        _, _, best_sd = find_best_timestep(params, Conv8, model_kwargs_c100, val_loader_c100, device='cpu')
        val_accs_c100[method_label] = run_subsequent_training(
            best_sd, Conv8, model_kwargs_c100,
            train_loader_c100, val_loader_c100, subsequent_cfg, device='cpu', label=method_label
        )
        gc.collect()
    else:
        print(f'{method_label} params not found. Rerun exp2c with params saved.') 

plot_subsequent_training(
    val_accs_per_method=val_accs_c100,
    title='CIFAR-10→CIFAR-100 - Subsequent Training (Fig. 6b)',
    save_path=os.path.join(RESULTS_DIR, 'figures', 'fig6b_cifar10_to_cifar100.png'),
    show=True
)
plt.close()
print('Figure 6b saved.')